[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/engineercarloscogua/task6-productionLLM/blob/main/notebooks/Actividad6_Entrega_Colab.ipynb)

# Actividad 6 - Llevando los LLM a producción

## Asistente de Fomento Socioeconómico UNITRÓPICO

**Caso de estudio:** agente conversacional con RAG sobre Renta Joven, Bienestar Contigo y la Política de Bienestar Universitario  
**Estudiantes:** Carlos Arnulfo Cogua Laverde, Michael Lan Alvarez y Marcela Gomez Garzon  
**Curso:** LLM - Maestría en Inteligencia Artificial - Universidad de La Salle  
**Repositorio:** https://github.com/engineercarloscogua/task6-productionLLM

Este notebook es el artefacto académico de entrega. Documenta y ejecuta las evidencias del desarrollo sin almacenar credenciales. La aplicación de usuario está construida con Streamlit; el notebook reproduce las pruebas, el RAG, las herramientas, el agente, la memoria y una prueba de rendimiento con diferentes cargas de trabajo.

> **Antes de entregar:** ejecuta `Entorno de ejecución > Ejecutar todas`, verifica que no existan errores, agrega la URL pública de Streamlit y descarga el notebook conservando sus salidas.


## 1. Objetivo y trazabilidad de la rúbrica

El objetivo es desplegar y evaluar un LLM integrado con LangChain para un caso de uso institucional, considerando funcionalidad, rendimiento, escalabilidad, ética y calidad de la comunicación.

| Criterio de evaluación | Puntos | Evidencia en este notebook |
|---|---:|---|
| Código estructurado y documentado | 1 | Secciones 2 y 3, módulos del repositorio y pruebas automatizadas |
| Agente, cadena, tools, RAG, memoria, rendimiento y cargas | 2 | Secciones 4 a 9 |
| Ética, privacidad y sesgos | 1 | Sección 11 |
| Resultados, interpretación y conclusiones | 1 | Secciones 10, 12 y 13 |

La interfaz se implementa con Streamlit, una alternativa equivalente al ejemplo de Gradio mencionado en la actividad. Google Colab se utiliza como entorno reproducible de documentación y evaluación; no se presenta como el servidor de producción.


## 2. Preparación reproducible en Google Colab

La siguiente celda detecta Colab, clona el repositorio público e instala las dependencias. En una ejecución local reutiliza el proyecto actual. No modifica ni imprime secretos.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/engineercarloscogua/task6-productionLLM.git'
REPO_DIR = Path('/content/task6-productionLLM')

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements-dev.txt')],
        check=True,
    )
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    REPO_DIR = next((path for path in candidates if (path / 'src').exists()), Path.cwd())

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Entorno:', 'Google Colab' if IN_COLAB else 'Local')
print('Proyecto:', REPO_DIR)
print('Python:', sys.version.split()[0])


## 3. Configuración segura del modelo

En Colab, crea el secreto `OPENROUTER_API_KEY` desde el panel **Secretos** y habilita su acceso al notebook. La clave se lee en memoria, no se imprime y nunca se guarda en Git. Si no existe, las pruebas deterministas continuarán funcionando y las evaluaciones contra OpenRouter quedarán marcadas como no ejecutadas.


In [ ]:
def read_openrouter_secret() -> str:
    if IN_COLAB:
        try:
            from google.colab import userdata  # type: ignore
            return str(userdata.get('OPENROUTER_API_KEY') or '').strip()
        except Exception:
            return ''
    return os.getenv('OPENROUTER_API_KEY', '').strip()

API_KEY = read_openrouter_secret()
if API_KEY:
    os.environ['OPENROUTER_API_KEY'] = API_KEY

from src.models import DEFAULT_MODEL_ID, fetch_free_tool_models

try:
    catalog = fetch_free_tool_models()
    available_ids = {option.id for option in catalog}
except Exception as error:
    catalog = []
    available_ids = set()
    print('Aviso: no se pudo actualizar el catálogo:', type(error).__name__)

MODEL_ID = DEFAULT_MODEL_ID if DEFAULT_MODEL_ID in available_ids else 'openrouter/free'
print('Clave disponible:', bool(API_KEY))
print('Modelo de evaluación:', MODEL_ID)
print('Modelos gratuitos con tools detectados:', len(catalog))


## 4. Arquitectura y decisiones técnicas

```text
Usuario
  -> interfaz Streamlit
      -> agente LangChain + memoria InMemorySaver
          -> consultar_documentos -> índice TF-IDF -> PDF con archivo y página
          -> calcular_transporte -> resultado determinista
      -> OpenRouter -> Nemotron 3.5 Lightning (free) o fallback disponible
```

La aplicación separa interfaz, modelo, agente, cadena LCEL, herramientas, recuperación y pruebas. Los PDF se extraen por página y se dividen en fragmentos solapados. El índice TF-IDF evita servicios de embeddings, aunque limita la recuperación semántica. Cada sesión de Streamlit utiliza un `thread_id` diferente para aislar su memoria conversacional.


In [ ]:
from src.agent import SYSTEM_PROMPT
from src.tools import TOOLS

print('PROMPT DE SISTEMA DEL AGENTE')
print(SYSTEM_PROMPT)
print('Herramientas disponibles:', [tool.name for tool in TOOLS])


## 5. Corpus documental y auditoría del RAG

Esta evidencia comprueba que los PDF contienen texto recuperable. Una página con texto no garantiza OCR perfecto; por ello se conservan archivo y página para verificación humana.


In [ ]:
from src.pdf_knowledge import PDF_DIRECTORY, pdf_knowledge_store

pdf_status = pdf_knowledge_store.refresh(force=True)
assert pdf_status.files, 'No se encontraron PDF en data/pdfs.'
assert pdf_status.chunks > 0, 'Los PDF no contienen texto extraíble.'

print('Archivos PDF:', len(pdf_status.files))
print('Páginas con texto:', pdf_status.pages_with_text)
print('Fragmentos indexados:', pdf_status.chunks)
print('Errores de extracción:', len(pdf_status.errors))
for filename in pdf_status.files:
    print(' -', filename)
if pdf_status.errors:
    print('Errores:', *pdf_status.errors, sep='\n - ')


## 6. Cadena LCEL

La rúbrica diferencia las cadenas de los agentes. Esta prueba construye una cadena fija `prompt -> modelo -> parser`. Se emplea un modelo falso determinista para que la evidencia sea reproducible y no consuma la API.


In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from src.chains import build_definition_chain

fake_model = FakeListChatModel(
    responses=['Un agente decide qué herramientas utilizar y una cadena sigue pasos predefinidos.']
)
definition_chain = build_definition_chain(fake_model)
chain_result = definition_chain.invoke({'concepto': 'un agente con herramientas'})
print(chain_result)
assert 'agente' in chain_result.lower()


## 7. RAG y herramientas

Se prueban las dos herramientas de manera independiente: recuperación documental con citas y cálculo determinista. Después, el agente decidirá cuándo utilizarlas.


In [ ]:
from src.tools import calcular_transporte, consultar_documentos

rag_queries = [
    '¿Cuáles son los requisitos para pertenecer a Renta Joven?',
    '¿Qué establece la Política de Bienestar Universitario?',
    '¿Qué fechas aparecen en la convocatoria pública 2026-1?',
]

for query in rag_queries:
    context = consultar_documentos.invoke({'pregunta': query})
    print('\nPREGUNTA:', query)
    print(context[:1800])
    assert 'Fuente:' in context and 'página' in context

transport_cost = calcular_transporte.invoke({
    'viajes_por_dia': 2,
    'dias': 5,
    'costo_por_viaje': 3000,
})
assert transport_cost == 30000
print('\nCálculo verificado:', transport_cost, 'pesos')


## 8. Agente, tool-calling y memoria conversacional

Esta sección realiza una prueba real si existe la clave. Los dos primeros turnos comparten el mismo `thread_id`, lo que demuestra memoria de corto plazo. El tercer turno exige la herramienta de cálculo. Se registran herramientas y latencia sin imprimir la credencial.


In [ ]:
live_agent_results = []

if API_KEY:
    from src.agent import build_agent, run_agent

    agent = build_agent(API_KEY, MODEL_ID)
    thread_id = 'colab-demostracion-memoria'
    prompts = [
        'Según los documentos, ¿qué es el programa Renta Joven?',
        '¿Cuáles son sus requisitos y en qué fuente aparecen?',
        'Calcula cuánto gasto si hago 2 viajes diarios durante 5 días a 3000 pesos por viaje.',
    ]

    for prompt in prompts:
        try:
            response = run_agent(agent, prompt, thread_id, MODEL_ID)
            row = {
                'pregunta': prompt,
                'respuesta': response.text,
                'herramientas': response.tools,
                'latencia_s': round(response.latency_seconds, 3),
                'estado': 'correcto',
            }
        except Exception as error:
            row = {
                'pregunta': prompt,
                'respuesta': '',
                'herramientas': [],
                'latencia_s': None,
                'estado': f'{type(error).__name__}: {str(error)[:250]}',
            }
        live_agent_results.append(row)
        print('\nPREGUNTA:', row['pregunta'])
        print('RESPUESTA:', row['respuesta'])
        print('TOOLS:', row['herramientas'], '| LATENCIA:', row['latencia_s'], '| ESTADO:', row['estado'])
else:
    print('NO EJECUTADO: configura OPENROUTER_API_KEY en Secretos de Colab.')


## 9. Pruebas automatizadas

La suite valida de forma determinista la recuperación, las citas, las herramientas, el catálogo, la cadena LCEL, el agente, la memoria y la lógica usada para resumir las pruebas de carga. Un resultado diferente de cero detiene la entrega.


In [ ]:
test_process = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests', '-q'],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
print(test_process.stdout)
if test_process.stderr:
    print(test_process.stderr)
assert test_process.returncode == 0, 'La suite de pruebas falló.'


## 10. Prueba de rendimiento y cargas de trabajo

Para evidenciar el criterio de la rúbrica se ejecutan tres solicitudes con carga baja, media y alta, representadas por concurrencias `1`, `2` y `5`. Se registran solicitudes correctas, fallidas y tiempo promedio de respuesta. Esta metodología es una decisión del equipo para observar el comportamiento bajo cargas diferentes; la rúbrica no impone niveles ni métricas específicas.

> La celda puede tardar varios minutos y consumir cuota del proveedor. Al utilizar un modelo `:free` no se esperan cargos por tokens, pero OpenRouter puede aplicar límites de solicitudes.


In [ ]:
RUN_LOAD_TEST = bool(API_KEY)
REQUESTS_PER_LEVEL = 3
summary_path = REPO_DIR / 'results' / 'benchmark_summary.json'
samples_path = REPO_DIR / 'results' / 'benchmark_samples.csv'

if RUN_LOAD_TEST:
    load_test_process = subprocess.run(
        [
            sys.executable, 'scripts/benchmark.py',
            '--model', MODEL_ID,
            '--concurrency', '1', '2', '5',
            '--requests-per-level', str(REQUESTS_PER_LEVEL),
        ],
        cwd=REPO_DIR,
        capture_output=True,
        text=True,
        env=os.environ.copy(),
    )
    print(load_test_process.stdout)
    if load_test_process.stderr:
        print(load_test_process.stderr)
    assert load_test_process.returncode == 0, 'La prueba de cargas no terminó correctamente.'
else:
    print('NO EJECUTADO: configura OPENROUTER_API_KEY para medir cargas reales.')


In [ ]:
load_test_summary = None
if summary_path.exists():
    load_test_summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('Fecha UTC:', load_test_summary['measured_at_utc'])
    print('Modelo:', load_test_summary['model'])
    print('| Carga concurrente | Solicitudes | Éxitos | Fallos | Tasa de éxito | Tiempo promedio (s) |')
    print('|---:|---:|---:|---:|---:|---:|')
    for group in load_test_summary['groups']:
        print(
            f"| {group['concurrency']} | {group['requests']} | {group['successes']} | "
            f"{group['failures']} | {group['success_rate']:.0%} | "
            f"{group['latency_mean_seconds']} |"
        )
else:
    print('SIN RESULTADOS: ejecuta la celda anterior con una clave válida.')


## 11. Ética, privacidad y sesgos

| Riesgo | Manifestación en este caso | Mitigación implementada | Riesgo residual |
|---|---|---|---|
| Alucinación | Inventar requisitos, fechas o beneficios | RAG, temperatura cero, prompt restrictivo y citas por archivo/página | El LLM aún puede interpretar mal un fragmento |
| Sesgo de cobertura | Los PDF no representan todas las necesidades estudiantiles | Respuestas limitadas al corpus y aviso cuando no existe evidencia | Un documento oficial también puede estar incompleto o desactualizado |
| Sesgo léxico | TF-IDF favorece coincidencias literales y perjudica sinónimos | Varias formulaciones de prueba y verificación de fuentes | Puede recuperar un fragmento poco pertinente |
| Privacidad | Los fragmentos recuperados se envían a OpenRouter | Solo documentos autorizados, secretos fuera de Git y memoria efímera | El proveedor procesa el contexto enviado |
| Exclusión por OCR | Errores de reconocimiento afectan nombres, tablas o cifras | Auditoría de páginas con texto y cita para revisión del PDF | Texto extraíble no implica OCR perfecto |
| Alto impacto | Una respuesta podría influir en beneficios estudiantiles | El sistema es informativo y exige verificación humana | No debe automatizar admisiones, sanciones o asignación de apoyos |
| Disponibilidad | Modelos gratuitos pueden desaparecer o devolver 429 | Catálogo actualizable, fallback y manejo explícito de errores | No existe garantía de servicio continuo |

La aplicación no toma decisiones sobre elegibilidad. Su función es facilitar la consulta de documentos, y la fuente oficial prevalece ante cualquier discrepancia.


## 12. Evidencia del despliegue Streamlit

La rúbrica solicita evidenciar el modelo en producción. La URL debe abrir desde una sesión externa y su endpoint de salud debe responder correctamente. Completa `STREAMLIT_APP_URL` cuando se realice el despliegue; una URL local no se considera evidencia pública.


In [ ]:
import requests

STREAMLIT_APP_URL = ''  # Ejemplo: https://nombre-app.streamlit.app
deployment_status = {'url': STREAMLIT_APP_URL, 'verified': False, 'detail': 'PENDIENTE'}

if STREAMLIT_APP_URL.strip():
    health_url = STREAMLIT_APP_URL.rstrip('/') + '/_stcore/health'
    try:
        health_response = requests.get(health_url, timeout=20)
        deployment_status = {
            'url': STREAMLIT_APP_URL,
            'verified': health_response.status_code == 200 and health_response.text.strip() == 'ok',
            'detail': f'HTTP {health_response.status_code}: {health_response.text[:100]}',
        }
    except Exception as error:
        deployment_status['detail'] = f'{type(error).__name__}: {str(error)[:200]}'

print(json.dumps(deployment_status, ensure_ascii=False, indent=2))


## 13. Resultados e interpretación

La interpretación debe apoyarse únicamente en las salidas reales de las celdas anteriores. El siguiente bloque resume la evidencia sin convertir una prueba local o una muestra pequeña en una garantía de escalabilidad.


In [ ]:
print('RESULTADOS CONSOLIDADOS')
print('- Corpus:', len(pdf_status.files), 'PDF,', pdf_status.pages_with_text, 'páginas con texto,', pdf_status.chunks, 'fragmentos.')
print('- Extracción:', 'sin errores' if not pdf_status.errors else f'{len(pdf_status.errors)} errores.')
print('- Cadena LCEL: ejecución determinista correcta.')
print('- RAG: tres consultas recuperaron fuente y página.')
print('- Herramienta de cálculo: resultado esperado de 30.000 pesos.')
print('- Pruebas automatizadas: aprobadas.' if test_process.returncode == 0 else '- Pruebas automatizadas: fallidas.')

if live_agent_results:
    live_successes = sum(row['estado'] == 'correcto' for row in live_agent_results)
    print(f'- Agente real: {live_successes}/{len(live_agent_results)} turnos correctos con {MODEL_ID}.')
else:
    print('- Agente real: NO EJECUTADO.')

if load_test_summary:
    groups = load_test_summary['groups']
    total_requests = sum(group['requests'] for group in groups)
    total_successes = sum(group['successes'] for group in groups)
    print(f'- Prueba de cargas: {total_successes}/{total_requests} solicitudes correctas.')
    baseline, highest = groups[0], groups[-1]
    if baseline['latency_mean_seconds'] is not None and highest['latency_mean_seconds'] is not None:
        print(
            f"- Interpretación de carga: el tiempo promedio pasó de {baseline['latency_mean_seconds']} s "
            f"con concurrencia {baseline['concurrency']} a {highest['latency_mean_seconds']} s "
            f"con concurrencia {highest['concurrency']}."
        )
    if any(group['failures'] for group in groups):
        print('- Se observaron fallos; la aplicación no demuestra estabilidad sostenida bajo todas las cargas.')
    else:
        print('- No hubo fallos en esta muestra; esto no garantiza disponibilidad futura.')
else:
    print('- Rendimiento y escalabilidad: NO MEDIDOS; no se formula una conclusión cuantitativa.')

print('- Despliegue público:', 'VERIFICADO' if deployment_status['verified'] else 'PENDIENTE O NO VERIFICADO.')


## 14. Conclusiones

1. La solución integra los componentes solicitados de LangChain: cadena LCEL, agente, dos herramientas, recuperación documental y memoria conversacional. La modularidad permite probar cada componente de forma independiente.
2. El RAG ofrece trazabilidad mediante archivo y página, pero la coincidencia TF-IDF y la calidad del OCR limitan la comprensión semántica. Las decisiones importantes deben verificarse directamente en los documentos oficiales.
3. Las pruebas deterministas demuestran corrección funcional del código, no calidad universal de las respuestas ni capacidad de producción sostenida. La interpretación de rendimiento debe basarse en los resultados observados con las cargas de trabajo seleccionadas.
4. La memoria es apropiada para continuidad dentro de una sesión, pero `InMemorySaver` no ofrece persistencia durable y pierde su estado al reiniciar el proceso.
5. El sistema debe mantenerse como apoyo informativo con revisión humana. Antes de considerarlo una solución institucional se requieren gobierno de datos, monitoreo, evaluación con usuarios, accesibilidad, persistencia y controles de seguridad adicionales.

**Criterio de cierre de la entrega:** el notebook solo debe presentarse como evidencia completa cuando la prueba de cargas tenga resultados reales, la URL pública esté verificada y todas las celdas se ejecuten sin errores.


## 15. Lista de verificación de entrega

Esta comprobación no reemplaza la revisión humana, pero evita presentar como completa una ejecución que aún tenga evidencias obligatorias pendientes.


In [ ]:
delivery_checks = {
    'Corpus PDF indexado': bool(pdf_status.files and pdf_status.chunks),
    'Cadena LCEL ejecutada': bool(chain_result),
    'RAG con fuentes y páginas': 'Fuente:' in context and 'página' in context,
    'Herramienta determinista validada': transport_cost == 30000,
    'Pruebas automatizadas aprobadas': test_process.returncode == 0,
    'Agente real ejecutado': bool(live_agent_results) and all(row['estado'] == 'correcto' for row in live_agent_results),
    'Prueba de cargas ejecutada': load_test_summary is not None,
    'Despliegue público verificado': deployment_status['verified'],
}

for item, passed in delivery_checks.items():
    print(('✅' if passed else '❌'), item)

DELIVERY_READY = all(delivery_checks.values())
print('\nLISTO PARA ENTREGAR:' if DELIVERY_READY else '\nAÚN NO ESTÁ LISTO PARA ENTREGAR:', DELIVERY_READY)


## Referencias

- Guía y rúbrica de la Actividad 6, suministradas en el aula virtual.
- LangChain. *Agents*. https://docs.langchain.com/oss/python/langchain/agents
- LangGraph. *Memory*. https://docs.langchain.com/oss/python/langgraph/add-memory
- OpenRouter. *Tool calling*. https://openrouter.ai/docs/features/tool-calling
- Streamlit. *Deploy your app*. https://docs.streamlit.io/deploy/streamlit-community-cloud/deploy-your-app
- Google Colaboratory. *FAQ*. https://research.google.com/colaboratory/faq.html
